# 01 — SKAB Data Exploration

Verifies that `src/preprocessing.py` loads SKAB correctly: shapes, summary stats, and a raw sensor plot with anomaly regions shaded red. This is the checkpoint before building features/models.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt

from src.preprocessing import load_anomaly_free, load_skab_single, load_skab_all, chronological_split

In [ ]:
anomaly_free = load_anomaly_free("../SKAB")
fault = load_skab_single("../SKAB/valve1/0.csv")

print("anomaly-free shape:", anomaly_free.shape)
print("valve1/0 shape:", fault.shape)
print("valve1/0 anomaly rate:", fault["anomaly"].mean())

In [ ]:
anomaly_free.describe()

In [ ]:
fault.describe()

In [ ]:
def anomaly_intervals(df):
    """Contiguous [start, end] timestamp ranges where anomaly == 1."""
    if df["anomaly"].sum() == 0:
        return []
    mask = df["anomaly"].astype(int)
    diff = mask.diff().fillna(0)
    starts = df.index[diff == 1].tolist()
    ends = df.index[diff == -1].tolist()
    if mask.iloc[0] == 1:
        starts.insert(0, df.index[0])
    if mask.iloc[-1] == 1:
        ends.append(df.index[-1])
    return list(zip(starts, ends))


sensor_cols = [c for c in fault.columns if c not in ("anomaly", "changepoint")]
intervals = anomaly_intervals(fault)

fig, axes = plt.subplots(len(sensor_cols), 1, figsize=(12, 2 * len(sensor_cols)), sharex=True)
for ax, col in zip(axes, sensor_cols):
    ax.plot(fault.index, fault[col], linewidth=0.8)
    for start, end in intervals:
        ax.axvspan(start, end, color="red", alpha=0.12)
    ax.set_ylabel(col, fontsize=8)
axes[-1].set_xlabel("datetime")
fig.suptitle("SKAB/valve1/0.csv — raw sensors, anomaly regions shaded red")
fig.tight_layout()
plt.show()

In [ ]:
fault_all = load_skab_all("../SKAB")
print("all fault CSVs shape:", fault_all.shape, "| distinct source files:", fault_all["source_file"].nunique())

train, test = chronological_split(fault_all)
print("train shape:", train.shape, "anomaly rate:", round(train["anomaly"].mean(), 3))
print("test shape:", test.shape, "anomaly rate:", round(test["anomaly"].mean(), 3))